In [7]:

import pandas as pd
import numpy as np

# =========================
# 1. LOAD DATA
# =========================
file_path = "/content/sample_data/time_series_60min_singleindex.csv"

df = pd.read_csv(file_path)

# =========================
# 2. FIX BROKEN COLUMN NAMES
# =========================
# convert u_t_c___t_i_m_e_s_t_a_m_p → utctimestamp
df.columns = [''.join(col.split('_')).lower() for col in df.columns]

# =========================
# 3. RENAME IMPORTANT COLUMNS
# =========================
df = df.rename(columns={
    'utctimestamp': 'timestamp',
    'atloadactualentsoetransparency': 'load_actual',
    'atloadforecastentsoetransparency': 'load_forecast',
    'atpricedayahead': 'price',
    'atsolargenerationactual': 'solar',
    'atwindonshoregenerationactual': 'wind'
})

# =========================
# 4. FIX TIMESTAMP (CRITICAL STEP)
# =========================
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

df = df.set_index('timestamp')

# =========================
# 5. SORT + ENSURE HOURLY CONTINUITY
# =========================
df = df.sort_index()

full_range = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq='H',
    tz='UTC'
)

df = df.reindex(full_range)

# =========================
# 6. HANDLE MISSING VALUES
# =========================
df = df.ffill()

# =========================
# 7. REMOVE DUPLICATES (SAFETY)
# =========================
df = df[~df.index.duplicated(keep='first')]

# =========================
# 8. KEEP ONLY AUSTRIA (REDUCE FEATURES)
# =========================
df = df[['load_actual', 'load_forecast', 'solar', 'wind']]

# =========================
# 9. OUTLIER TREATMENT
# =========================
def cap_outliers(col):
    Q1 = col.quantile(0.25)
    Q3 = col.quantile(0.75)
    IQR = Q3 - Q1
    return np.clip(col, Q1 - 1.5*IQR, Q3 + 1.5*IQR)

for col in df.columns:
    df[col] = cap_outliers(df[col])

# =========================
# 10. FEATURE ENGINEERING
# =========================
df['hour'] = df.index.hour
df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month

# lag features
df['load_lag_1'] = df['load_actual'].shift(1)
df['load_lag_24'] = df['load_actual'].shift(24)

df = df.ffill()

# =========================
# 11. FINAL OUTPUT
# =========================
df = df.reset_index()

df.to_csv("austria_clean_final.csv", index=False)

print("Data cleaned successfully")
print("Shape:", df.shape)
print(df.head())

/tmp/ipykernel_32970/1759532539.py:41: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_range = pd.date_range(


✅ Data cleaned successfully
Shape: (50401, 10)
                      index  load_actual  load_forecast  solar  wind  hour  \
0 2014-12-31 23:00:00+00:00          NaN            NaN    NaN   NaN    23   
1 2015-01-01 00:00:00+00:00       5946.0         6701.0    NaN  69.0     0   
2 2015-01-01 01:00:00+00:00       5726.0         6593.0    NaN  64.0     1   
3 2015-01-01 02:00:00+00:00       5347.0         6482.0    NaN  65.0     2   
4 2015-01-01 03:00:00+00:00       5249.0         6454.0    NaN  64.0     3   

   dayofweek  month  load_lag_1  load_lag_24  
0          2     12         NaN          NaN  
1          3      1         NaN          NaN  
2          3      1      5946.0          NaN  
3          3      1      5726.0          NaN  
4          3      1      5347.0          NaN  
